# setup + imports

In [1]:
import random
import numpy as np
import pandas as pd
import torch


In [6]:
animals = [line.strip() for line in open('animals_pool.txt') if line.strip()]
question_data = pd.read_csv('questions_pool.csv')
table = pd.read_csv('oracle_table.csv', dtype=str)

qids = list(question_data['question_id'])
names = list(table['animal'])

mat = np.zeros((len(names), len(qids)), dtype=int)
for i in range(len(names)):
    rasp = table['responses'][i]
    for q in range(len(qids)):
        mat[i][q] = int(rasp[q])
print(mat.shape)

(250, 280)


In [12]:
cnt = 0

def newid():
    global cnt
    i = cnt
    cnt = cnt + 1
    return i

rows = []
def build(cands):
    node = newid()

    if len(cands) <= 2:
        now = node
        for k in range(len(cands)):
            if k == len(cands)-1:
                nxt = 'END'
            else:
                nxt = newid()
            row = 'G|' + names[cands[k]] + '|' + str(nxt)
            rows.append((now, row))
            now = nxt                                                                        
        return node

    bestq = 0
    bestscore = -1
    for q in range(len(qids)):
        nryes = 0
        for i in cands:
            if mat[i][q] == 1:
                nryes += 1

        nrno = len(cands) - nryes
        score = min(nryes, nrno)
        
        if score > bestscore:
            bestscore = score
            bestq = q
    
    yes = [i for i in cands if mat[i][bestq]==1]
    no = [i for i in cands if mat[i][bestq]==0]
    yesnode = build(yes)
    nonode = build(no)
    row = "Q|" + qids[bestq] + '|' + str(yesnode) + '|' + str(nonode)
    rows.append((node, row))
    return node
    
    


In [13]:
build(range(len(names)))

0

In [14]:
import zipfile
rows.sort()
sub = pd.DataFrame({'id': [r[0] for r in rows], 'subtaskID': 1, 'answer': [r[1] for r in rows]})
sub.to_csv('submission.csv', index=False)
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('submission.csv')